In [1]:
import pandas as pd
import joblib
import numpy as np

In [2]:
# Load saved objects
num_imputer = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/num_imputer.pkl')
cat_imputer = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/cat_imputer.pkl')
scaler = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/scaler.pkl')
model = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/final_model_tuned.pkl')
ord_encoder = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/ordinal_encoder.pkl')
ohe = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/onehot_encoder.pkl')


In [3]:
# Selected features after feature importance
selected_features = [
    'oldpeak','thalach','ca',
    'sex_1.0','cp_2.0','cp_3.0','cp_4.0',
    'fbs_1.0','exang_1.0','thal_6.0','thal_7.0'
]

# Columns from training
categorical_cols = ["sex","cp","fbs","restecg","exang","slope","ca","thal"]
ordinal_cols = ["slope","restecg","ca"]
nominal_cols = ["sex","cp","fbs","exang","thal"]
numeric_cols = ["age","trestbps","chol","thalach","oldpeak"]

In [4]:
def preprocess_new_data(new_data):
    # --- Step 1: Impute ---
    new_data[numeric_cols] = num_imputer.transform(new_data[numeric_cols])
    new_data[categorical_cols] = cat_imputer.transform(new_data[categorical_cols])

    # --- Step 2: Encoding ---
    # Ordinal encoding
    new_data[ordinal_cols] = ord_encoder.transform(new_data[ordinal_cols])

    # One-hot encoding for nominal
    nominal_encoded = ohe.transform(new_data[nominal_cols])
    nominal_encoded_df = pd.DataFrame(
        nominal_encoded,
        columns=ohe.get_feature_names_out(nominal_cols),
        index=new_data.index
    )

    # Drop nominal & concat encoded
    new_data = new_data.drop(columns=nominal_cols)
    new_data = pd.concat([new_data, nominal_encoded_df], axis=1)

    # --- Step 3: Scaling ---
    new_data[numeric_cols] = scaler.transform(new_data[numeric_cols])

    # --- Step 4: Select features (same order as training) ---
    for col in selected_features:
        if col not in new_data.columns:
            new_data[col] = 0  # add if missing
    new_data = new_data[selected_features]

    return new_data

In [17]:
# Example patients
test_patients = pd.DataFrame([
    {   # Patient 1: Healthy (likely No Heart Disease)
        'age': 40,         # younger age
    'sex': 0,          # female
    'cp': 1,           # typical angina
    'trestbps': 120,   # normal resting BP
    'chol': 200,       # healthy cholesterol
    'fbs': 0,          # normal fasting blood sugar
    'restecg': 0,      # normal ECG
    'thalach': 170,    # high maximum heart rate (good)
    'exang': 0,        # no exercise-induced angina
    'oldpeak': 0.0,    # no ST depression
    'slope': 2,        # upsloping ST (good)
    'ca': 0,           # no major vessels colored
    'thal': 3
    },
    {   # Patient 2: Diseased (likely Heart Disease)
        'age': 60,
    'sex': 1,
    'cp': 3,
    'trestbps': 130,
    'chol': 250,
    'fbs': 1,
    'restecg': 0,
    'thalach': 140,
    'exang': 0,
    'oldpeak': 2.3,
    'slope': 1,
    'ca': 0,
    'thal': 7
    }
])

In [18]:
# Preprocess and predict
X_test_processed = preprocess_new_data(test_patients)
predictions = model.predict(X_test_processed)

# Show results
for i, pred in enumerate(predictions):
    print(f"Patient {i+1} prediction:", "Heart Disease" if pred == 1 else "No Heart Disease")

Patient 1 prediction: No Heart Disease
Patient 2 prediction: Heart Disease
